# 11 — Conventional probabilistic baselines
This notebook adds a benchmark ladder to the completed financial experiment without refitting a neural model: a centered historical joint bootstrap, a fixed-covariance Gaussian VAR, and a VAR with marginal GARCH(1,1) scales plus joint standardized-residual resampling. All use the same training standardization and 957 test targets as notebook 08.

In [1]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from innovcal.baselines import diagnose_var_garch, forecast_baselines
from innovcal.data.windows import Standardizer, chronological_split
from innovcal.evaluation import evaluate_samples, paired_seed_block_bootstrap
OUT = ROOT / 'results' / 'conventional_baselines'; OUT.mkdir(parents=True, exist_ok=True)

## Fit training-only baselines
The VAR lag is the only selected baseline hyperparameter and is chosen by validation one-step MSE from the frozen set {1, 5, 10, 20}. VAR and GARCH parameters remain training-only. At test origin t, the conditional variance uses residuals only through t−1.

## VAR–GARCH specification diagnostics
These checks use the training partition only. VAR stability is assessed from the companion-matrix spectral radius. Marginal GARCH admissibility is summarized by parameter positivity, optimizer convergence, and persistence $\alpha+\beta<1$. Ljung–Box checks at lags 5, 10, and 20 are applied to standardized residuals and their squares; their p-values are descriptive because they do not adjust for parameter estimation. Heavy-tailed standardized residuals are not automatically a model failure because the forecast resamples complete residual vectors rather than imposing Gaussian innovations.

In [2]:
ASSETS = ['AAPL', 'JPM', 'XOM', 'WMT']
returns = pd.read_csv(ROOT / 'data' / 'processed' / 'financial_returns.csv', index_col=0, parse_dates=True)[ASSETS].dropna().sort_index()
split = chronological_split(returns.to_numpy())
scaler = Standardizer.fit(split.train)
train, validation, test = map(scaler.transform, (split.train, split.validation, split.test))
baselines = forecast_baselines(train, validation, test, lag_candidates=(1, 5, 10, 20), n_samples=500, seed=20_606)
print('Selected VAR lag:', baselines.selected_lag)
display(pd.Series(baselines.validation_mse, name='validation_mse').rename_axis('lag').to_frame())
print('GARCH optimizer convergence by asset:', dict(zip(ASSETS, baselines.garch_converged)))
diagnostics = diagnose_var_garch(train, baselines.selected_lag, ljung_box_lags=(5, 10, 20))
print(f'VAR companion spectral radius: {diagnostics.var_spectral_radius:.6f}')
print('VAR stable (spectral radius < 1):', diagnostics.var_stable)
garch_diagnostics = pd.DataFrame({
    'asset': ASSETS,
    'omega': diagnostics.garch.omega,
    'alpha': diagnostics.garch.alpha,
    'beta': diagnostics.garch.beta,
    'persistence': diagnostics.garch.alpha + diagnostics.garch.beta,
    'optimizer_converged': diagnostics.garch.converged,
    'variance_clipping_rate': diagnostics.variance_clipping_rates,
    'std_residual_skewness': diagnostics.residual_skewness,
    'std_residual_kurtosis': diagnostics.residual_kurtosis,
    'jarque_bera_pvalue': diagnostics.jarque_bera_pvalues,
})
garch_diagnostics['stationary_garch'] = garch_diagnostics['persistence'] < 1.0
display(garch_diagnostics)
assert diagnostics.var_stable, 'Selected VAR is dynamically unstable.'
assert garch_diagnostics['optimizer_converged'].all(), 'At least one GARCH fit used the fallback parameters.'
assert garch_diagnostics['stationary_garch'].all(), 'At least one GARCH marginal is nonstationary.'

Selected VAR lag: 1


,validation_mse
lag,
1,1.231921
5,1.264987
10,1.304973
20,1.336326


GARCH optimizer convergence by asset: {'AAPL': np.True_, 'JPM': np.True_, 'XOM': np.True_, 'WMT': np.True_}
VAR companion spectral radius: 0.151321
VAR stable (spectral radius < 1): True


,asset,omega,alpha,beta,persistence,optimizer_converged,variance_clipping_rate,std_residual_skewness,std_residual_kurtosis,jarque_bera_pvalue,stationary_garch
0,AAPL,0.028277,0.088958,0.881037,0.969995,True,0.0,-0.136476,6.298713,1.746865e-283,True
1,JPM,0.011133,0.117392,0.867500,0.984892,True,0.0,-0.233115,5.534992,3.864360e-172,True
2,XOM,0.013202,0.090419,0.894459,0.984879,True,0.0,-0.353561,4.878656,1.122755e-104,True
3,WMT,0.011878,0.044972,0.944472,0.989444,True,0.0,-0.247149,11.535926,0.000000e+00,True


In [3]:
ljung_box_rows = []
for lag_index, lag in enumerate(diagnostics.ljung_box_lags):
    for asset_index, asset in enumerate(ASSETS):
        ljung_box_rows.append({
            'asset': asset,
            'lag': int(lag),
            'standardized_residual_pvalue': diagnostics.residual_ljung_box_pvalues[lag_index, asset_index],
            'squared_standardized_residual_pvalue': diagnostics.squared_residual_ljung_box_pvalues[lag_index, asset_index],
        })
ljung_box_diagnostics = pd.DataFrame(ljung_box_rows)
display(ljung_box_diagnostics)
residual_correlation = pd.DataFrame(
    diagnostics.standardized_residual_correlation, index=ASSETS, columns=ASSETS
)
display(residual_correlation.style.format('{:.3f}').background_gradient(cmap='coolwarm', vmin=-1, vmax=1))
print('Ljung–Box rejections at 5% (standardized residuals):',
      int((ljung_box_diagnostics['standardized_residual_pvalue'] < 0.05).sum()),
      'of', len(ljung_box_diagnostics))
print('Ljung–Box rejections at 5% (squared standardized residuals):',
      int((ljung_box_diagnostics['squared_standardized_residual_pvalue'] < 0.05).sum()),
      'of', len(ljung_box_diagnostics))
print('Non-Gaussian marginals at 5% by Jarque–Bera:',
      int((garch_diagnostics['jarque_bera_pvalue'] < 0.05).sum()),
      'of', len(garch_diagnostics))
garch_diagnostics.assign(
    var_spectral_radius=diagnostics.var_spectral_radius,
    var_stable=diagnostics.var_stable,
).to_csv(OUT / 'var_garch_parameter_diagnostics.csv', index=False)
ljung_box_diagnostics.to_csv(OUT / 'var_garch_ljung_box_diagnostics.csv', index=False)
residual_correlation.to_csv(OUT / 'standardized_residual_correlation.csv')

,asset,lag,standardized_residual_pvalue,squared_standardized_residual_pvalue
0,AAPL,5,0.236879,0.869711
1,JPM,5,0.038992,0.328214
2,XOM,5,0.001688,0.494801
3,WMT,5,0.090212,0.595104
4,AAPL,10,0.072972,0.920320
5,JPM,10,0.073968,0.382661
6,XOM,10,0.024795,0.492054
7,WMT,10,0.003492,0.727347
8,AAPL,20,0.230736,0.968891
9,JPM,20,0.149040,0.736997


,AAPL,JPM,XOM,WMT
AAPL,1.000,0.363,0.316,0.221
JPM,0.363,1.000,0.489,0.324
XOM,0.316,0.489,1.000,0.303
WMT,0.221,0.324,0.303,1.000


Ljung–Box rejections at 5% (standardized residuals): 6 of 12
Ljung–Box rejections at 5% (squared standardized residuals): 0 of 12
Non-Gaussian marginals at 5% by Jarque–Bera: 4 of 4


### Diagnostic interpretation
A spectral radius below one supports dynamic stability of the fitted VAR. Persistence below one supports covariance stationarity of each fitted GARCH process; values close to one indicate slowly decaying volatility. Small Ljung–Box p-values for standardized residuals indicate remaining linear dynamics, while small values for squared standardized residuals indicate remaining volatility dependence. Jarque–Bera rejection documents non-Gaussian marginal shape and supports empirical rather than Gaussian innovation resampling. Nonzero off-diagonal residual correlations justify resampling synchronized residual vectors instead of sampling each asset independently.

In [4]:
# Baseline fitting and diagnostics are completed in the preceding cells.

In [5]:
archive = np.load(ROOT / 'results' / 'seed_uncertainty' / 'seed_forecasts.npz')
manifest = pd.read_csv(ROOT / 'results' / 'seed_uncertainty' / 'seed_forecast_manifest.csv')
neural = {row.model: archive[row.array_key] for row in manifest.itertuples()}
target, projections = archive['target'], archive['projections']
assert target.shape == baselines.target.shape == test.shape
assert np.allclose(target, baselines.target, atol=1e-6)
baseline_metrics = pd.DataFrame([{'model': name, **evaluate_samples(target, samples, projections=projections)} for name, samples in baselines.samples.items()])
neural_metrics = pd.read_csv(ROOT / 'results' / 'seed_uncertainty' / 'seed_metrics.csv').groupby('model', as_index=False).mean(numeric_only=True)
display(pd.concat([neural_metrics, baseline_metrics], ignore_index=True)[['model', 'rmse', 'energy_score', 'interval_score', 'coverage', 'interval_width', 'pit_calibration_error', 'pit_mean_absolute_autocorrelation']])

,model,rmse,energy_score,interval_score,coverage,interval_width,pit_calibration_error,pit_mean_absolute_autocorrelation
0,CA-RNN,0.950396,1.156251,4.119090,0.866641,2.493779,0.001306,0.038585
1,CA-RNN sequential,0.947946,1.153081,4.081602,0.876959,2.576625,0.001225,0.037910
2,RNN,0.939856,1.145041,4.085468,0.871682,2.489145,0.001438,0.031359
3,Historical bootstrap,0.938597,1.152985,4.309647,0.900470,2.823129,0.001234,0.028595
4,VAR Gaussian,0.944386,1.176671,4.399342,0.927900,3.249237,0.003388,0.037944
5,VAR-GARCH bootstrap,0.945201,1.149505,4.029825,0.898903,2.754430,0.000453,0.038422


## Dependence-aware comparisons
Each conventional forecast is repeated across the ten neural-seed slots. Consequently, these intervals incorporate forecast-origin dependence and neural optimization variability but condition on the single fitted baseline. Candidate-minus-comparator differences below zero favor the candidate for loss metrics.

In [6]:
n_seeds = neural['RNN'].shape[0]
combined = dict(neural)
combined.update({name: np.repeat(samples[None], n_seeds, axis=0) for name, samples in baselines.samples.items()})
comparisons = []
for candidate in ['RNN', 'CA-RNN sequential']:
    comparisons.extend((candidate, baseline) for baseline in baselines.samples)
inference = paired_seed_block_bootstrap(target, combined, comparisons, projections, block_length=20, n_bootstrap=2000, confidence=0.95, seed=1108)
inference['resolved'] = (inference.ci_lower > 0) | (inference.ci_upper < 0)
display(inference[inference.metric.isin(['energy_score', 'interval_score', 'coverage_error', 'pit_calibration_error'])][['candidate', 'comparator', 'metric', 'difference', 'ci_lower', 'ci_upper', 'resolved']])

,candidate,comparator,metric,difference,ci_lower,ci_upper,resolved
1,RNN,Historical bootstrap,energy_score,-0.007944,-0.010591,-0.004669,True
2,RNN,Historical bootstrap,interval_score,-0.224179,-0.275878,-0.177810,True
5,RNN,Historical bootstrap,coverage_error,0.027847,0.012220,0.025256,True
6,RNN,Historical bootstrap,pit_calibration_error,0.000204,0.000054,0.000313,True
9,RNN,VAR Gaussian,energy_score,-0.031630,-0.034994,-0.028435,True
10,RNN,VAR Gaussian,interval_score,-0.313874,-0.361304,-0.273255,True
13,RNN,VAR Gaussian,coverage_error,0.000418,-0.008674,0.009536,False
14,RNN,VAR Gaussian,pit_calibration_error,-0.001949,-0.002278,-0.001625,True
17,RNN,VAR-GARCH bootstrap,energy_score,-0.004464,-0.007043,-0.002727,True
18,RNN,VAR-GARCH bootstrap,interval_score,0.055643,0.022684,0.076751,True


In [7]:
baseline_metrics.assign(selected_var_lag=baselines.selected_lag).to_csv(OUT / 'baseline_metrics.csv', index=False)
pd.DataFrame({'asset': ASSETS, 'garch_optimizer_converged': baselines.garch_converged}).to_csv(OUT / 'garch_convergence.csv', index=False)
inference.to_csv(OUT / 'baseline_hierarchical_inference.csv', index=False)
payload = {'target': target, 'projections': projections, **{name.replace(' ', '_').replace('-', '_'): value for name, value in baselines.samples.items()}}
np.savez_compressed(OUT / 'baseline_forecasts.npz', **payload)
print('Saved baseline metrics, forecasts, convergence status, and paired inference.')

Saved baseline metrics, forecasts, convergence status, and paired inference.


## Interpretation guardrails
A CA-RNN calibration advantage over Gaussian VAR but not VAR–GARCH would indicate that volatility adaptation, rather than recurrent nonlinear dynamics, explains the gain. An advantage over VAR–GARCH on PIT calibration accompanied by worse Energy Score is a calibration–accuracy tradeoff, not dominance. Historical bootstrap is a low-information reference and should not be treated as the principal baseline.